# 01 — Data Exploration
Explore a SOLEY batch output: schema, fault distribution,
weather patterns, and the engineered features used by ML models.


## 1. Imports & setup

In [1]:
import sys, pathlib

# Ensure both repo root and its parent are importable, regardless of where
# the notebook kernel starts.
cwd = pathlib.Path.cwd()
sys.path.insert(0, str(cwd))
sys.path.insert(0, str(cwd.parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from library.utils import setup_logging
from library.config import BatchConfig
from library.data.loader import load_split
from library.data import prepare_file_registry
from library.features import add_features

setup_logging()
%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 2. Point to your data
Set `DATA_DIR` to the folder that contains your SOLEY parquet files
(and optionally `batch_config.json`).


In [2]:
DATA_DIR   = "dataset"
OUTPUT_DIR = "outputs/exploration"
import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


## 3. Auto-detect configuration

In [3]:
cfg = BatchConfig(DATA_DIR)
print(cfg)
print()
print(f"Array capacity  : {cfg.array_kwp:.2f} kWp")
print(f"Locations       : {len(cfg.locations)}")
print(f"SCADA features  : {len(cfg.scada_features)}")
print(f"IV-curve features: {len(cfg.iv_curve_features)}")
print(f"Stress features : {len(cfg.stress_features)}")
print(f"Total features  : {len(cfg.all_feature_cols)}")


  Config: array=5.34 kWp, 9 location(s) from batch_config.json
  Simulation-only columns found in data and excluded from all feature sets: ['detailed_balance_efficiency_pct'] (theoretical-only), ['ac_wiring_loss_factor', 'aging_factor', 'aoi_correction_factor', 'availability_factor', 'bifacial_gain', 'dc_wiring_loss_factor', 'inverter_derating_factor', 'inverter_efficiency', 'inverter_loss_kw', 'lid_factor', 'mismatch_loss_factor', 'shading_factor', 'snow_loss_factor', 'soiling_factor', 'spectral_factor', 'vulnerability'] (simulator-internal — never a training input, even for "full")
  Columns: 11 SCADA, 6 IV-curve, 5 stress, 38 constant (excluded), 17 simulation-only (excluded), 0 uncategorized (excluded), 22 total features
BatchConfig(array_kwp=5.34, locations=9, features=22)

Array capacity  : 5.34 kWp
Locations       : 9
SCADA features  : 11
IV-curve features: 6
Stress features : 5
Total features  : 22


## 4. Load a sample of data (no subsampling limit)

In [ ]:
# Build a temporary registry for exploration — no split assignment
# needed since we're just sampling the data, not training.
# Using prepare_file_registry + load_split ensures the same
# preprocessing (daytime filter, feature engineering) as training.
registry_explore = prepare_file_registry(DATA_DIR, cfg)
df_explore = load_split(registry_explore, cfg, max_rows="auto")
df_explore["timestamp"] = pd.to_datetime(df_explore["timestamp"])
print(df_explore.shape)
df_explore.head(3)


Registry: 418 files, 12 fault types
After daytime filter: 298,919 rows
After daytime filter: 306,524 rows
After daytime filter: 306,632 rows
  estimate_row_budget: 739 bytes/row (incl. 3.0x safety margin), 50.6 GB available, 40% budgeted → cap = 29,409,903 rows
After daytime filter: 298,919 rows
After daytime filter: 306,524 rows
After daytime filter: 306,632 rows
After daytime filter: 306,789 rows
After daytime filter: 307,284 rows
After daytime filter: 306,679 rows
After daytime filter: 303,994 rows
After daytime filter: 303,982 rows
After daytime filter: 305,354 rows
After daytime filter: 306,769 rows
After daytime filter: 307,267 rows
After daytime filter: 306,798 rows
After daytime filter: 307,837 rows
After daytime filter: 306,838 rows
After daytime filter: 306,825 rows
After daytime filter: 306,794 rows
After daytime filter: 306,782 rows
After daytime filter: 306,605 rows
After daytime filter: 306,795 rows
After daytime filter: 306,808 rows
After daytime filter: 306,872 rows
Aft

## 5. Schema & dtypes

In [ ]:
df_explore.dtypes.to_frame("dtype")


## 6. Fault distribution

In [ ]:
print("fault_active counts")
print(df_explore["fault_active"].value_counts())
print()
print("fault_type counts (faulted rows only)")
faulted = df_explore[df_explore["fault_active"]]
vc = faulted["fault_type"].value_counts()
print(vc)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fault prevalence
ax = axes[0]
labels = ["Healthy", "Faulted"]
sizes  = [len(df_explore) - len(faulted), len(faulted)]
ax.pie(sizes, labels=labels, autopct="%1.1f%%",
       colors=["#10b981", "#dc2626"], startangle=90)
ax.set_title("Fault Prevalence")

# Fault type breakdown
ax = axes[1]
vc.plot(kind="barh", ax=ax, color="#2563eb")
ax.set_xlabel("Row count")
ax.set_title("Fault Type Distribution (faulted rows)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fault_distribution.png", bbox_inches="tight")
plt.show()


## 7. Temporal overview

In [ ]:
if "sim_year" in df_explore.columns:
    year_counts = df_explore.groupby("sim_year")["fault_active"].agg(["sum", "count"])
    year_counts["fault_rate"] = year_counts["sum"] / year_counts["count"]
    display(year_counts)


## 8. Apply feature engineering & inspect new features

In [ ]:
df = add_features(df_explore.copy(), cfg.array_kwp)
new_cols = ["hour_sin", "hour_cos", "doy_sin", "doy_cos",
            "performance_ratio", "pr_deviation",
            "dc_ac_power_ratio", "power_step"]
df[new_cols].describe().round(4)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for ax, col in zip(axes, new_cols):
    healthy = df[~df["fault_active"]][col].dropna()
    bad     = df[ df["fault_active"]][col].dropna()
    ax.hist(healthy, bins=50, alpha=0.6, label="Healthy", color="#10b981",
            density=True)
    ax.hist(bad,     bins=50, alpha=0.6, label="Faulted", color="#dc2626",
            density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle("Engineered Feature Distributions: Healthy vs. Faulted",
             fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_distributions.png", bbox_inches="tight")
plt.show()


## 9. Correlation heatmap (SCADA features)

In [ ]:
import seaborn as sns

scada_cols = [c for c in cfg.scada_features if c in df.columns]
corr = df[scada_cols + ["fault_active"]].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap="RdBu_r", center=0,
            linewidths=0.3, ax=ax)
ax.set_title("SCADA Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/correlation_heatmap.png", bbox_inches="tight")
plt.show()


## 10. Sample time-series: one run

In [ ]:
if "sim_day" in df.columns:
    # Pick one day from one run
    sample = df[df["sim_day"] == df["sim_day"].iloc[100]].copy()
    sample = sample.sort_values("timestamp")

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    axes[0].plot(sample["timestamp"], sample["poa_global_wm2"],
                 color="#f59e0b", label="POA irradiance")
    axes[0].set_ylabel("W/m²"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(sample["timestamp"], sample["ac_power_kw"],
                 color="#2563eb", label="AC power")
    axes[1].set_ylabel("kW"); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(sample["timestamp"], sample["performance_ratio"],
                 color="#10b981", label="Performance Ratio")
    # Shade faulted periods
    if sample["fault_active"].any():
        fa = sample["fault_active"].values
        ts = sample["timestamp"].values
        axes[2].fill_between(ts, 0, 1,
                             where=fa, alpha=0.25, color="red",
                             label="Fault active", transform=axes[2].get_xaxis_transform())
    axes[2].set_ylabel("PR"); axes[2].legend(); axes[2].grid(alpha=0.3)

    fig.suptitle("Sample Time-Series", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/sample_timeseries.png", bbox_inches="tight")
    plt.show()
else:
    print("sim_day column not present — skipping time-series plot.")
